<a href="https://colab.research.google.com/github/ashivashankars/CMPE258_Assignments/blob/main/24_weights_and_biases.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Notebook 24 — Weights & Biases Experiment Tracking

## What This Notebook Covers
As models and experiments grow, tracking results in print statements
becomes unmanageable. Weights & Biases (W&B) is the industry-standard
experiment tracking platform. It automatically logs metrics,
hyperparameters, model artifacts, system usage, and predictions.

**Topics covered:**
- `wandb.init()` — start a run and log config
- `wandb.log()` — log metrics per epoch/step
- `wandb.summary` — log best values for a run
- `wandb.watch()` — auto-log model gradients and weights
- `wandb.Artifact` — version model checkpoints
- `wandb.Table` — log predictions for visual inspection
- W&B Sweeps — automated hyperparameter search
- Keras callback integration
- PyTorch custom loop integration
- Multi-run comparison

**Dataset:** Fashion-MNIST


In [ ]:
!pip install wandb --quiet

import wandb
import tensorflow as tf
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

tf.random.set_seed(42)
torch.manual_seed(42)
np.random.seed(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('wandb      :', wandb.__version__)
print('TensorFlow :', tf.__version__)
print('PyTorch    :', torch.__version__)
print('Device     :', device)

wandb      : 0.26.1
TensorFlow : 2.20.0
PyTorch    : 2.10.0+cu128
Device     : cuda


## 1. Login to W&B

Get your API key from https://wandb.ai/authorize
Use `anonymous='allow'` to run without an account (limited features).


In [ ]:
# Option 1: interactive login
# wandb.login()

# Option 2: anonymous mode — no account needed
wandb.login(anonymous='allow')
print('W&B ready. View runs at: https://wandb.ai')

wandb: WARNING The anonymous parameter to wandb.login() has no effect and will be removed in future versions.
/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:

 3


wandb: You chose "Don't visualize my results"
wandb: Using W&B in offline mode.
wandb: W&B API key is configured. Use `wandb login --relogin` to force relogin


W&B ready. View runs at: https://wandb.ai


## 2. Load Data


In [ ]:
(X_train_full, y_train_full), (X_test, y_test) = tf.keras.datasets.fashion_mnist.load_data()
X_train_full = X_train_full.astype('float32') / 255.0
X_test       = X_test.astype('float32')       / 255.0
X_train, X_valid = X_train_full[5000:], X_train_full[:5000]
y_train, y_valid = y_train_full[5000:], y_train_full[:5000]

CLASS_NAMES = ['T-shirt','Trouser','Pullover','Dress','Coat',
               'Sandal','Shirt','Sneaker','Bag','Ankle boot']

transform = transforms.Compose([
    transforms.ToTensor(), transforms.Normalize((0.5,), (0.5,))
])
train_full_pt = datasets.FashionMNIST('./data', train=True,  download=True, transform=transform)
test_set_pt   = datasets.FashionMNIST('./data', train=False, download=True, transform=transform)
train_pt, valid_pt = torch.utils.data.random_split(
    train_full_pt, [55000, 5000], generator=torch.Generator().manual_seed(42)
)
train_loader = DataLoader(train_pt,    batch_size=64, shuffle=True,  pin_memory=True)
valid_loader = DataLoader(valid_pt,    batch_size=64, shuffle=False, pin_memory=True)
test_loader  = DataLoader(test_set_pt, batch_size=64, shuffle=False, pin_memory=True)
print('Data ready.')

29515/29515 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
26421880/26421880 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
5148/5148 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
4422102/4422102 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


100%|██████████| 26.4M/26.4M [00:02<00:00, 12.0MB/s]
100%|██████████| 29.5k/29.5k [00:00<00:00, 210kB/s]
100%|██████████| 4.42M/4.42M [00:01<00:00, 3.79MB/s]
100%|██████████| 5.15k/5.15k [00:00<00:00, 20.3MB/s]

Data ready.


## 3. Model Factories


In [ ]:
def make_tf_model(hidden_units=256, dropout_rate=0.3):
    return tf.keras.Sequential([
        tf.keras.layers.Flatten(input_shape=(28, 28)),
        tf.keras.layers.Dense(hidden_units, activation='relu',
                               kernel_initializer='he_normal'),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.Dropout(dropout_rate),
        tf.keras.layers.Dense(hidden_units // 2, activation='relu',
                               kernel_initializer='he_normal'),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.Dropout(dropout_rate * 0.7),
        tf.keras.layers.Dense(10, activation='softmax'),
    ])


class FashionNetPT(nn.Module):
    def __init__(self, hidden_units=256, dropout_rate=0.3):
        super().__init__()
        self.net = nn.Sequential(
            nn.Flatten(),
            nn.Linear(784, hidden_units),
            nn.BatchNorm1d(hidden_units), nn.ReLU(),
            nn.Dropout(dropout_rate),
            nn.Linear(hidden_units, hidden_units // 2),
            nn.BatchNorm1d(hidden_units // 2), nn.ReLU(),
            nn.Dropout(dropout_rate * 0.7),
            nn.Linear(hidden_units // 2, 10)
        )
    def forward(self, x): return self.net(x)

print('Model factories ready.')

Model factories ready.


---
## Part A — TensorFlow + W&B

## 4. Basic Run — wandb.init, wandb.log, wandb.config

Every experiment starts with `wandb.init()`. This creates a unique run,
logs the config, and connects to W&B servers.
Then `wandb.log({...})` sends metrics to the live dashboard.


In [ ]:
config = {
    'learning_rate': 1e-3,
    'hidden_units':  256,
    'dropout_rate':  0.3,
    'batch_size':    64,
    'epochs':        15,
    'optimizer':     'adam',
    'framework':     'tensorflow',
}

run = wandb.init(
    project='deep-learning-assignment',
    name='tf-baseline',
    config=config,
    tags=['tensorflow', 'baseline'],
    notes='Baseline TF model — MLP on Fashion-MNIST',
)

print(f'Run name: {run.name}')
print(f'Run URL : {run.url}')

wandb: WARNING URL not available in offline run


Run name: tf-baseline
Run URL : None


In [ ]:
# Always read from wandb.config so sweeps can override values
cfg = wandb.config

tf.random.set_seed(42)
model_tf = make_tf_model(cfg.hidden_units, cfg.dropout_rate)
model_tf.compile(
    optimizer=tf.keras.optimizers.Adam(cfg.learning_rate),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

best_val_acc = 0.0

for epoch in range(cfg.epochs):
    h = model_tf.fit(
        X_train, y_train,
        validation_data=(X_valid, y_valid),
        initial_epoch=epoch,
        epochs=epoch + 1,
        batch_size=cfg.batch_size,
        verbose=0
    )
    t_acc  = h.history['accuracy'][-1]
    v_acc  = h.history['val_accuracy'][-1]
    t_loss = h.history['loss'][-1]
    v_loss = h.history['val_loss'][-1]
    best_val_acc = max(best_val_acc, v_acc)

    # Log metrics — these appear as live curves on the W&B dashboard
    wandb.log({
        'epoch':      epoch + 1,
        'train/loss': t_loss,
        'train/acc':  t_acc,
        'val/loss':   v_loss,
        'val/acc':    v_acc,
        'lr':         float(model_tf.optimizer.learning_rate),
    })

    if (epoch + 1) % 5 == 0:
        print(f'Epoch {epoch+1:2d} | train={t_acc:.4f} | val={v_acc:.4f}')

# Summary — shown in the runs table
wandb.summary['best_val_accuracy'] = best_val_acc
wandb.finish()
print(f'Run complete. Best val acc: {best_val_acc:.4f}')

/usr/local/lib/python3.12/dist-packages/keras/src/layers/reshaping/flatten.py:37: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Epoch  5 | train=0.8643 | val=0.8632
Epoch 10 | train=0.8758 | val=0.8802
Epoch 15 | train=0.8865 | val=0.8802


epoch,▁▁▂▃▃▃▄▅▅▅▆▇▇▇█
lr,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
train/acc,▁▄▅▅▆▆▆▇▇▇▇▇███
train/loss,█▅▄▃▃▃▂▂▂▂▂▁▁▁▁
val/acc,▁▁▁▁▄▃▆▃▆▇▅▇██▇
val/loss,█▆█▇▅▅▄▅▃▂▃▃▁▂▂
best_val_accuracy,0.883
epoch,15
lr,0.001
train/acc,0.88655
train/loss,0.30757


Run complete. Best val acc: 0.8830


## 5. Keras Callback Integration

`wandb.keras.WandbCallback` automatically logs all metrics from
`model.fit()`, learning rate changes, and optionally weight histograms.
This is the simplest way to add W&B to an existing Keras project.


In [ ]:
run2 = wandb.init(
    project='deep-learning-assignment',
    name='tf-keras-callback',
    config={
        'learning_rate': 5e-4,
        'hidden_units':  512,
        'dropout_rate':  0.4,
        'batch_size':    64,
        'epochs':        15,
    },
    tags=['tensorflow', 'keras-callback'],
)

tf.random.set_seed(42)
model_tf2 = make_tf_model(wandb.config.hidden_units, wandb.config.dropout_rate)
model_tf2.compile(
    optimizer=tf.keras.optimizers.Adam(wandb.config.learning_rate),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

model_tf2.fit(
    X_train, y_train,
    validation_data=(X_valid, y_valid),
    epochs=wandb.config.epochs,
    batch_size=wandb.config.batch_size,
    callbacks=[
        wandb.keras.WandbMetricsLogger(), # Replaced deprecated WandbCallback
        tf.keras.callbacks.EarlyStopping(patience=5, restore_best_weights=True),
        tf.keras.callbacks.ReduceLROnPlateau(factor=0.5, patience=3, verbose=0),
    ],
    verbose=0
)

best2 = max(model_tf2.history.history['val_accuracy'])
wandb.summary['best_val_accuracy'] = best2
wandb.finish()
print(f'Keras callback run best val acc: {best2:.4f}')

/usr/local/lib/python3.12/dist-packages/keras/src/layers/reshaping/flatten.py:37: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


epoch/accuracy,▁▄▅▅▆▆▆▆▆▇▇████
epoch/epoch,▁▁▂▃▃▃▄▅▅▅▆▇▇▇█
epoch/learning_rate,█████████▁▁▁▁▁▁
epoch/loss,█▅▄▄▃▃▃▃▂▂▂▁▁▁▁
epoch/val_accuracy,▁▃▄▄▅▆▆▆▅▇▇▇███
epoch/val_loss,█▆▅▅▃▃▄▃▄▂▁▂▁▁▁
best_val_accuracy,0.8958
epoch/accuracy,0.89958
epoch/epoch,14
epoch/learning_rate,0.00025
epoch/loss,0.26921


Keras callback run best val acc: 0.8958


## 6. Log Artifacts and Prediction Table


In [ ]:
run3 = wandb.init(
    project='deep-learning-assignment',
    name='tf-artifacts-and-table',
    config={'epochs': 5, 'learning_rate': 1e-3,
            'hidden_units': 256, 'dropout_rate': 0.3, 'batch_size': 64}
)

tf.random.set_seed(42)
model_art = make_tf_model(wandb.config.hidden_units, wandb.config.dropout_rate)
model_art.compile(
    optimizer=tf.keras.optimizers.Adam(wandb.config.learning_rate),
    loss='sparse_categorical_crossentropy', metrics=['accuracy']
)
model_art.fit(X_train, y_train, validation_data=(X_valid, y_valid),
               epochs=wandb.config.epochs, batch_size=wandb.config.batch_size,
               verbose=0)

# Save model and log as artifact
model_art.save('/tmp/fashion_artifact.keras')
artifact = wandb.Artifact(
    name='fashion-mnist-tf-model',
    type='model',
    description='MLP trained on Fashion-MNIST',
    metadata={'val_acc': max(model_art.history.history['val_accuracy'])}
)
artifact.add_file('/tmp/fashion_artifact.keras')
run3.log_artifact(artifact)

# Log prediction table — visualize per-image predictions in W&B
preds_proba = model_art.predict(X_test[:50], verbose=0)
preds_class = preds_proba.argmax(axis=1)

table = wandb.Table(columns=['image', 'true_label', 'predicted', 'confidence', 'correct'])
for i in range(50):
    img_rgb = (np.stack([X_test[i]] * 3, axis=-1) * 255).astype(np.uint8)
    table.add_data(
        wandb.Image(img_rgb),
        CLASS_NAMES[y_test[i]],
        CLASS_NAMES[preds_class[i]],
        float(preds_proba[i].max()),
        bool(preds_class[i] == y_test[i])
    )
wandb.log({'predictions': table})

# Log confusion matrix
all_preds = model_art.predict(X_test, verbose=0).argmax(axis=1)
wandb.log({
    'confusion_matrix': wandb.plot.confusion_matrix(
        probs=None,
        y_true=y_test.tolist(),
        preds=all_preds.tolist(),
        class_names=CLASS_NAMES
    )
})

wandb.summary['best_val_accuracy'] = max(model_art.history.history['val_accuracy'])
wandb.finish()
print('Artifact and prediction table logged.')

/usr/local/lib/python3.12/dist-packages/keras/src/layers/reshaping/flatten.py:37: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


best_val_accuracy,0.8744


Artifact and prediction table logged.


## 7. W&B Sweeps — Automated Hyperparameter Search


In [ ]:
sweep_config = {
    'method': 'random',
    'metric': {'name': 'val/acc', 'goal': 'maximize'},
    'parameters': {
        'learning_rate': {'distribution': 'log_uniform_values',
                          'min': 1e-4, 'max': 1e-2},
        'hidden_units':  {'values': [128, 256, 512]},
        'dropout_rate':  {'distribution': 'uniform', 'min': 0.1, 'max': 0.5},
        'batch_size':    {'values': [32, 64, 128]},
    }
}


def sweep_train_fn():
    """
    Called by wandb.agent() for each sweep trial.
    wandb.config is auto-populated with the trial's hyperparameters.
    """
    run = wandb.init(project='deep-learning-assignment')
    cfg = wandb.config

    tf.random.set_seed(42)
    model = make_tf_model(cfg.hidden_units, cfg.dropout_rate)
    model.compile(
        optimizer=tf.keras.optimizers.Adam(cfg.learning_rate),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )

    best_val = 0.0
    for epoch in range(10):
        h = model.fit(
            X_train, y_train,
            validation_data=(X_valid, y_valid),
            initial_epoch=epoch, epochs=epoch+1,
            batch_size=cfg.batch_size, verbose=0
        )
        v_acc   = h.history['val_accuracy'][-1]
        best_val = max(best_val, v_acc)
        wandb.log({'val/acc': v_acc, 'epoch': epoch+1})

    wandb.summary['best_val_accuracy'] = best_val
    wandb.finish()


sweep_id = wandb.sweep(sweep_config, project='deep-learning-assignment')
print(f'Sweep created: {sweep_id}')
wandb.agent(sweep_id, function=sweep_train_fn, count=5)
print('Sweep complete. View results in W&B dashboard.')

wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:

 3


wandb: You chose "Don't visualize my results"
wandb: Using W&B in offline mode.
wandb: ERROR Error while calling W&B API: api key too short (<Response [401]>)
wandb: ERROR Error while calling W&B API: api key too short (<Response [401]>)
wandb: ERROR Error while calling W&B API: api key too short (<Response [401]>)
wandb: ERROR Error while calling W&B API: api key too short (<Response [401]>)
wandb: ERROR Error while calling W&B API: api key too short (<Response [401]>)


CommError: 'errors'

---
## Part B — PyTorch + W&B

## 8. PyTorch Custom Loop with Full W&B Integration


In [ ]:
run_pt = wandb.init(
    project='deep-learning-assignment',
    name='pytorch-custom-loop',
    config={
        'learning_rate': 1e-3,
        'hidden_units':  256,
        'dropout_rate':  0.3,
        'epochs':        15,
        'batch_size':    64,
        'framework':     'pytorch'
    },
    tags=['pytorch']
)
cfg_pt = wandb.config

torch.manual_seed(42)
pt_model   = FashionNetPT(cfg_pt.hidden_units, cfg_pt.dropout_rate).to(device)
pt_opt     = torch.optim.Adam(pt_model.parameters(), lr=cfg_pt.learning_rate)
pt_sched   = torch.optim.lr_scheduler.CosineAnnealingLR(pt_opt, T_max=cfg_pt.epochs)
pt_crit    = nn.CrossEntropyLoss()

# wandb.watch() auto-logs gradients and weights every log_freq steps
wandb.watch(pt_model, pt_crit, log='all', log_freq=100)

best_val_pt  = 0.0
best_ckpt_pt = '/tmp/pt_wandb_best.pt'

for epoch in range(cfg_pt.epochs):
    pt_model.train()
    t_correct = t_total = 0
    t_loss_sum = 0.0
    grad_norms = []

    for X_b, y_b in train_loader:
        X_b, y_b = X_b.to(device), y_b.to(device)
        pt_opt.zero_grad()
        out  = pt_model(X_b)
        loss = pt_crit(out, y_b)
        loss.backward()
        gnorm = nn.utils.clip_grad_norm_(pt_model.parameters(), 1.0)
        grad_norms.append(float(gnorm))
        pt_opt.step()
        t_correct  += (out.detach().argmax(1) == y_b).sum().item()
        t_total    += y_b.size(0)
        t_loss_sum += loss.item() * X_b.size(0)

    pt_model.eval()
    v_correct = v_total = 0
    v_loss_sum = 0.0
    with torch.no_grad():
        for X_b, y_b in valid_loader:
            X_b, y_b = X_b.to(device), y_b.to(device)
            out  = pt_model(X_b)
            loss = pt_crit(out, y_b)
            v_correct  += (out.argmax(1) == y_b).sum().item()
            v_total    += y_b.size(0)
            v_loss_sum += loss.item() * X_b.size(0)

    t_acc = t_correct  / t_total
    t_lss = t_loss_sum / t_total
    v_acc = v_correct  / v_total
    v_lss = v_loss_sum / v_total
    mean_gnorm = float(np.mean(grad_norms))
    curr_lr    = pt_opt.param_groups[0]['lr']

    wandb.log({
        'epoch':         epoch + 1,
        'train/loss':    t_lss,
        'train/acc':     t_acc,
        'val/loss':      v_lss,
        'val/acc':       v_acc,
        'grad_norm':     mean_gnorm,
        'learning_rate': curr_lr,
    })

    if v_acc > best_val_pt:
        best_val_pt = v_acc
        torch.save(pt_model.state_dict(), best_ckpt_pt)

    pt_sched.step()

    if (epoch + 1) % 5 == 0:
        print(f'Epoch {epoch+1:2d} | train={t_acc:.4f} | val={v_acc:.4f} | '
              f'gnorm={mean_gnorm:.3f} | lr={curr_lr:.2e}')

# Log prediction table
pt_model.load_state_dict(torch.load(best_ckpt_pt, map_location=device))
pt_model.eval()

X_samp, y_samp_t = next(iter(test_loader))
X_samp = X_samp[:32].to(device)
y_samp = y_samp_t[:32].numpy()

with torch.no_grad():
    probs_samp = torch.softmax(pt_model(X_samp), dim=1).cpu().numpy()
preds_samp = probs_samp.argmax(axis=1)

pred_table = wandb.Table(columns=['image','true','predicted','confidence','correct'])
X_np = X_samp.cpu().numpy()
for i in range(32):
    img_u8  = ((X_np[i, 0] * 0.5 + 0.5) * 255).clip(0, 255).astype(np.uint8)
    img_rgb = np.stack([img_u8] * 3, axis=-1)
    pred_table.add_data(
        wandb.Image(img_rgb),
        CLASS_NAMES[y_samp[i]],
        CLASS_NAMES[preds_samp[i]],
        float(probs_samp[i].max()),
        bool(preds_samp[i] == y_samp[i])
    )
wandb.log({'predictions_table': pred_table})

# Log model artifact
art = wandb.Artifact('pt-fashion-model', type='model',
                      metadata={'best_val_acc': best_val_pt})
art.add_file(best_ckpt_pt)
run_pt.log_artifact(art)

wandb.summary['best_val_accuracy'] = best_val_pt
wandb.finish()
print(f'\nPyTorch run best val acc: {best_val_pt:.4f}')

## 9. Multi-Run Comparison


In [ ]:
variants = [
    {'name': 'small-256',  'hidden_units': 256,  'learning_rate': 1e-3,  'dropout_rate': 0.2},
    {'name': 'medium-512', 'hidden_units': 512,  'learning_rate': 3e-4,  'dropout_rate': 0.3},
    {'name': 'large-1024', 'hidden_units': 1024, 'learning_rate': 1e-3,  'dropout_rate': 0.4},
]

local_results = []

for v in variants:
    run_v = wandb.init(
        project='deep-learning-assignment',
        name=v['name'],
        config={**v, 'epochs': 10, 'batch_size': 64},
        tags=['comparison', 'pytorch']
    )
    cfg_v = wandb.config

    torch.manual_seed(42)
    m_v   = FashionNetPT(cfg_v.hidden_units, cfg_v.dropout_rate).to(device)
    o_v   = torch.optim.Adam(m_v.parameters(), lr=cfg_v.learning_rate)
    s_v   = torch.optim.lr_scheduler.CosineAnnealingLR(o_v, T_max=cfg_v.epochs)
    c_v   = nn.CrossEntropyLoss()
    best_v = 0.0
    accs_v = []

    for epoch in range(cfg_v.epochs):
        m_v.train()
        for X_b, y_b in train_loader:
            X_b, y_b = X_b.to(device), y_b.to(device)
            o_v.zero_grad()
            c_v(m_v(X_b), y_b).backward()
            o_v.step()

        m_v.eval()
        vc = vt = 0
        with torch.no_grad():
            for X_b, y_b in valid_loader:
                X_b, y_b = X_b.to(device), y_b.to(device)
                vc += (m_v(X_b).argmax(1) == y_b).sum().item()
                vt += y_b.size(0)
        v_acc = vc / vt
        best_v = max(best_v, v_acc)
        accs_v.append(v_acc)
        wandb.log({'val/acc': v_acc, 'epoch': epoch+1})
        s_v.step()

    wandb.summary['best_val_accuracy'] = best_v
    wandb.finish()
    local_results.append((v['name'], best_v, accs_v))
    print(f'{v["name"]:15s} -> best val acc: {best_v:.4f}')

# Local comparison plot
plt.figure(figsize=(10, 4))
for (name, best, accs), color in zip(local_results, ['steelblue', 'tomato', 'seagreen']):
    plt.plot(accs, color=color, linewidth=1.8, label=f'{name} ({best:.4f})')
plt.title('Multi-run comparison (also on W&B dashboard)')
plt.xlabel('Epoch')
plt.ylabel('Validation Accuracy')
plt.legend(fontsize=9)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---
## Summary

### W&B Core API Cheatsheet

```python
import wandb

# 1. Start run
run = wandb.init(
    project='my-project',
    name='run-name',
    config={'lr': 1e-3, 'epochs': 20},
    tags=['baseline']
)

# 2. Log per-step metrics
wandb.log({'val/acc': 0.89, 'val/loss': 0.32, 'epoch': 5})

# 3. Log run-level summary
wandb.summary['best_val_acc'] = 0.92

# 4. Watch model (auto-log gradients)
wandb.watch(model, log='all', log_freq=100)

# 5. Log artifact (versioned model checkpoint)
art = wandb.Artifact('model', type='model')
art.add_file('model.pt')
run.log_artifact(art)

# 6. Log prediction table
table = wandb.Table(columns=['image', 'true', 'pred', 'confidence'])
table.add_data(wandb.Image(img), 'cat', 'dog', 0.72)
wandb.log({'predictions': table})

# 7. Finish
wandb.finish()
```

### Sweep Config Template
```python
sweep_config = {
    'method': 'bayes',
    'metric': {'name': 'val/acc', 'goal': 'maximize'},
    'parameters': {
        'lr':      {'distribution': 'log_uniform_values', 'min': 1e-4, 'max': 1e-2},
        'units':   {'values': [128, 256, 512]},
        'dropout': {'distribution': 'uniform', 'min': 0.1, 'max': 0.5},
    }
}
sweep_id = wandb.sweep(sweep_config, project='my-project')
wandb.agent(sweep_id, function=train_fn, count=20)
```

### What W&B Tracks Automatically
- System metrics: GPU/CPU usage, memory, temperature
- Environment: Python version, installed packages, git commit
- Code: the exact script that ran each experiment
- Everything logged via `wandb.log()` and `wandb.watch()`

**This completes Part 2 — all 24 notebooks are done!**
